# OneVoice V2.1 — sinh English construction audio
Sinh trực tiếp English clean + 2 noisy variants/mỗi utterance vào Google Drive. Không cần chạy data audit V1 trước. Generator tự resume từ `manifest.jsonl`; trạng thái ở `generation_state.json` để đổi GPU hoặc restart Colab không mất tiến độ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
OUTPUT_ROOT = MYDRIVE / 'onevoice_audio_v2_1'
CACHE_ROOT = MYDRIVE / 'OneVoice/model_cache'
for path in (OUTPUT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['PYTHONUNBUFFERED'] = '1'
os.chdir(REPO)
print('Output:', OUTPUT_ROOT)


In [ ]:
# None = toàn bộ 8.064 English utterances. Đặt 10 để smoke-test trước nếu muốn.
MAX_UTTERANCES = None
SAMPLES_PER_TEXT = 2  # 8.064 clean + 16.128 noisy khi chạy toàn bộ
os.environ['ONEVOICE_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
os.environ['ONEVOICE_DATA_DIR'] = str(REPO / 'data/onevoice_construction_v2')
os.environ['ONEVOICE_LANGUAGES'] = 'en'
os.environ['ONEVOICE_SAMPLES_PER_TEXT'] = str(SAMPLES_PER_TEXT)
if MAX_UTTERANCES is None:
    os.environ.pop('ONEVOICE_MAX_UTTERANCES', None)
else:
    os.environ['ONEVOICE_MAX_UTTERANCES'] = str(MAX_UTTERANCES)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy', 'pandas', 'soundfile', 'librosa', 'edge-tts', 'nest_asyncio', 'pydub', 'tqdm'], check=True)
command = [sys.executable, 'scripts/generate_synthetic_audio_v2_1.py']
print('>', ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code:
    raise RuntimeError(f'Generator stopped with exit code {code}. Re-run this cell: completed files remain on Drive and generation resumes.')


In [ ]:
import json
STATE = OUTPUT_ROOT / 'generation_state.json'
MANIFEST = OUTPUT_ROOT / 'manifest.jsonl'
if STATE.is_file():
    display(json.loads(STATE.read_text(encoding='utf-8')))
if MANIFEST.is_file():
    lines = sum(1 for _ in MANIFEST.open(encoding='utf-8'))
    print('Manifest noisy entries:', lines)
print('Clean WAV:', len(list((OUTPUT_ROOT / 'clean').glob('*_en_clean.wav'))))
print('Noisy WAV:', len(list((OUTPUT_ROOT / 'noisy').glob('*_en_n*.wav'))))
